# Parses and builds the metadata for all recorded videos in the videos directory using 

https://abhitronix.github.io/deffcode/latest
https://abhitronix.github.io/deffcode/latest/recipes/basic/

In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
%matplotlib qt5
import mne
mne.viz.set_browser_backend("qt")  # or "matplotlib"
mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

# ==================================================================================================================================================================================================================================================================================== #
# PyQtInspect:                                                                                                                                                                                                                                                                         #
# ==================================================================================================================================================================================================================================================================================== #
# 1. Launch `pqi-server` in a new terminal BEFORE running this notebook cell. Be sure to click 'Serve' button in the GUI that appears so this notebook can connect.

# # IMPORTANT: Call settrace BEFORE importing PyQt5
# import PyQtInspect.pqi as pqi

# # Connect to the server (default: localhost:19394)
# # Make sure the server is already running!
# pqi.settrace(
#     host='127.0.0.1',
#     port=19394,  # Default port, or use the port shown in the server GUI
#     qt_support='pyqt5',  # or 'auto' for auto-detection
#     patch_multiprocessing=False
# )

# # # !pip install viztracer
# %load_ext viztracer
# from viztracer import VizTracer


import holoviews as hv
hv.extension('bokeh', logo=False)

import hvplot.pandas
# This line is crucial for displaying plots in a notebook
hvplot.extension('bokeh') # You can also use 'matplotlib' or 'plotly'

# hv.extension('bokeh')
# hv.extension('matplotlib') # or 'matplotlib'
# hv.extension('plotly') # or 'matplotlib'
import panel as pn
pn.extension()

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import time

from typing import Dict, List, Tuple, Any

from pathlib import Path
import numpy as np
import pandas as pd

from mne import set_log_level
from copy import deepcopy
import mne

datasets = []
# mne.viz.set_browser_backend("Matplotlib")
mne.viz.set_browser_backend("qt")


from phopymnehelper.historical_data import HistoricalData
from phopymnehelper.SavedSessionsProcessor import SavedSessionsProcessor, SessionModality, DataModalityType


def get_now_time_str(time_separator='-') -> str:
    return str(time.strftime(f"%Y-%m-%d_%H{time_separator}%m", time.localtime(time.time())))


set_log_level("WARNING")


# db_root_path = Path('/content/drive/MyDrive/Databases')
# db_root_path = Path(r'E:/Dropbox (Personal)/Databases') ## APOGEE
db_root_path = Path(r'E:/Dropbox (Personal)/Databases') # WIN10_VM
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

# eeg_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/fif')
# headset_motion_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif')

# assert eeg_recordings_file_path.exists()
# assert headset_motion_recordings_file_path.exists()

eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/fif')
flutter_eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings')
flutter_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/MOTION_RECORDINGS')
flutter_GENERIC_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/GENERIC_RECORDINGS')

headset_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif')
WhisperVideoTranscripts_LSL_Converted = db_root_path.joinpath('UnparsedData/WhisperVideoTranscripts_LSL_Converted')
pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs')
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()

# lab_recorder_output_path = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001")
lab_recorder_output_path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001')
assert lab_recorder_output_path.exists()

# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
# n_most_recent_sessions_to_preprocess: int = 35
# n_most_recent_sessions_to_preprocess: int = 5
n_most_recent_sessions_to_preprocess: int = 10
# n_most_recent_sessions_to_preprocess = None

# # modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=lab_recorder_output_path, recordings_extensions=['.xdf'])
# modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], recordings_extensions=['.xdf']) ## both sources
# # modern_found_EEG_recording_files

# most_recent_modern_found_EEG_recording_files: List[Path] = modern_found_EEG_recording_files[:n_most_recent_sessions_to_preprocess]
# # most_recent_modern_found_EEG_recording_files

Automatic pdb calling has been turned OFF
Using qt as 2D backend.


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\utils\mixins\time_slicing.py:403: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class TimePointEventAccessor(TimeColumnAliasesProtocol, TimeSlicableObjectProtocol):


Using matplotlib as 2D backend.


C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoPyMNEHelper\src\phopymnehelper\helpers\indexing_helpers.py:1080: UserWarning: registration of accessor <class 'phopymnehelper.helpers.indexing_helpers.PhoDataframeAccessor'> under name 'pho' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class PhoDataframeAccessor:


TabError: inconsistent use of tabs and spaces in indentation (manager.py, line 54)

## Timeline

In [ ]:
import pyphoplacecellanalysis.External.pyqtgraph as pg
from pypho_timeline.timeline_builder import TimelineBuilder
# from pypho_timeline.widgets import SimpleTimelineWidget, perform_process_all_streams
# from pypho_timeline.__main__ import PositionTrackDatasource, VideoTrackDatasource, main, main_all_modalities_from_xdf_file_example

# Create Qt application
app = pg.mkQApp("pyPhoTimelineVideosOnlyExample")

builder: TimelineBuilder = TimelineBuilder()


# Timeline Widget from just Video Track

In [ ]:
from pathlib import Path
from pypho_timeline.rendering.datasources.specific.video import VideoTrackDatasource
from pypho_timeline.rendering.graphics.track_renderer import TrackRenderer

max_num_video_files: int = 1000

# Specify your video folder path
# video_folder = Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/Videos")
video_folder = Path(r"M:/ScreenRecordings/EyeTrackerVR_Recordings")
assert video_folder.exists() and video_folder.is_dir()

# Gather all video files (adjust extensions as needed)
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.wmv')
all_videos = [p for p in video_folder.glob('*') if p.suffix.lower() in video_extensions]

# Sort by modification time (descending), get 5 most recent
all_videos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
recent_videos = all_videos[:max_num_video_files]
# recent_videos



# Create the VideoTrackDatasource
video_ds: VideoTrackDatasource = VideoTrackDatasource(video_paths=recent_videos)

# Choose a name for the new video track
video_track_name: str = "RecentVideosTrack"



In [ ]:
parsed_video_out_parent_path = Path(r'C:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\PhoOfflineEEGAnalysis\output')
parsed_video_out_file_path: Path = video_ds.save_metadata_csv(parsed_video_out_parent_path=parsed_video_out_parent_path)
parsed_video_out_file_path

In [ ]:
parsed_video_out_file_path: Path = Path(r'C:/Users/pho/repos/EmotivEpoc/ACTIVE_DEV/PhoOfflineEEGAnalysis/output/2026-03-01_parsed_videos.csv')
loaded_video_ds: VideoTrackDatasource = VideoTrackDatasource.init_from_saved_metadata_csv(parsed_video_out_file_path=parsed_video_out_file_path)
loaded_video_ds

In [ ]:
video_only_timeline = builder.build_from_video(video_datasource=video_ds) # , video_paths=recent_videos


In [ ]:
video_ds.total_df_start_end_times
video_metadata_df: pd.DataFrame = deepcopy(video_ds.df).drop(columns=['pen', 'brush', 'series_vertical_offset', 'series_height'], inplace=False)
video_metadata_df


In [ ]:
# parsed_video_out_path: Path = Path(r'C:\Users\pho\repos\ACTIVE_DEV\PhoOfflineEEGAnalysis\output').resolve()
parsed_video_out_path: Path = Path(r'C:/Users/pho/repos/EmotivEpoc/ACTIVE_DEV/PhoOfflineEEGAnalysis/output').resolve()
assert parsed_video_out_path.exists() and parsed_video_out_path.is_dir()
parsed_video_out_file_path = parsed_video_out_path.joinpath(f'2026-02-24_parsed_videos.csv')
print(f'writing video metadata csv out to "{parsed_video_out_file_path}"...')
video_metadata_df.to_csv(parsed_video_out_file_path)
print(f'\tdone.')

In [ ]:

# Add the new video track to the existing timeline
video_track_name: str = 'VideoTrack'
# timeline.add_video_track(video_track_name, video_ds)
video_track_widget, video_track_root_graphics, video_track_plot_item, video_track_dock = timeline.add_video_track(track_name=video_track_name, video_datasource=video_ds)
video_track_renderer: TrackRenderer = timeline.track_renderers[video_track_name]
video_widget, video_track_renderer, video_track_datasource = timeline.get_track_tuple(video_track_name)

# timeline.add_track(video_ds, name=video_track_name)


In [ ]:
video_track_renderer: TrackRenderer = video_only_timeline.track_renderers['VideoTrack']
video_widget, video_track_renderer, video_track_datasource = video_only_timeline.get_track_tuple('VideoTrack')

In [ ]:
# video_track_renderer.detail_graphics
video_track_renderer.detail_renderer # video_track_renderer

# video_widget.

In [ ]:
video_metadata_df

In [ ]:

from pypho_timeline.rendering.datasources.specific.video import VideoDeffcodeHelpers



In [ ]:
acitve_video_metadata_df = video_track_datasource.video_metadata_df.tail(5)

# acitve_video_metadata_df = video_metadata_df.tail(5)
acitve_video_metadata_df

built_metadata = {}
generated_thumbnails = {}
for a_row in acitve_video_metadata_df.itertuples():
    a_path = Path(a_row.video_file_path)
    if a_path.exists():
        primary_vid_metadata, vid_metadata = VideoDeffcodeHelpers.fetch_video_metadata_for_cache(a_video_file=a_path, debug_log_metadata=False)
        built_metadata[a_path] = vid_metadata
        # frame = fetch_video_metadata_and_thumbnail_for_cache(a_video_file=a_path, save_output_thumbnail=False)
        frame = VideoDeffcodeHelpers.fetch_video_metadata_and_thumbnail_for_cache(a_video_file=a_path, save_output_thumbnail=False)

        generated_thumbnails[a_path] = frame
    else:
        generated_thumbnails[a_path] = None

## OUTPUTS: generated_thumbnails
# generated_thumbnails
built_metadata

In [ ]:
# built_metadata
# sourcer.probe_stream()
# included_metadata_fields = ['source_extension', 'source_video_resolution', 'source_video_pixfmt', 'source_video_framerate', 'source_video_orientation', 'source_video_decoder', 'source_duration_sec', 'approx_video_nframes', 'source_video_bitrate', 'source_audio_bitrate', 'source_audio_samplerate', 'source_has_video', 'source_has_audio', 'source_has_image_sequence']
included_primary_metadata_fields = ['source_video_framerate', 'source_duration_sec', 'approx_video_nframes']
included_secondary_metadata_fields = ['source_extension', 'source_video_resolution', 'source_video_pixfmt', 'source_video_orientation', 'source_video_decoder']
advanced_metadata_fields = ['source_video_bitrate', 'source_audio_bitrate', 'source_audio_samplerate', 'source_has_video', 'source_has_audio', 'source_has_image_sequence']

# print(list(sourcer.retrieve_metadata().keys()))
sourcer.retrieve_metadata()

In [ ]:
generated_thumbnails

In [ ]:
from pypho_timeline.rendering.graphics.track_renderer import TrackRenderer

# video_only_timeline.track_datasources[video_track_name]
# video_ds

video_track_renderer: TrackRenderer = video_only_timeline.track_renderers['VideoTrack']
video_track_renderer

### Add "now" lines

In [ ]:
import pyqtgraph as pg
from datetime import datetime
from pypho_timeline.utils.datetime_helpers import datetime_to_unix_timestamp

# Get current datetime
now_dt = datetime.now()

# Convert to unix timestamp
now_timestamp = datetime_to_unix_timestamp(now_dt)

# Create a thick red pen
red_pen = pg.mkPen(color='red', width=3)

now_line_items = {}
# Add the vertical line to all plot items
for plot_item in timeline.interval_rendering_plots:
    vline = pg.InfiniteLine(angle=90, movable=False, pos=now_timestamp)
    vline.setPen(red_pen)
    plot_item.addItem(vline, ignoreBounds=True)
    now_line_items[plot_item] = vline
    


## Export loaded XDF data for analysis in an external apps

#### To .json

In [ ]:
from phopymnehelper.exporters.JSON_Exporter import export_xdf_data_to_json
from pathlib import Path

output_path = Path("xdf_export.json")
export_xdf_data_to_json(
    eeg_raws=_out_eeg_raw,
    stream_infos_df=_out_xdf_stream_infos_df,
    output_path=output_path,
    include_raw_data=True,
    max_samples_per_stream=10000  # Optional: limit for large files
)



In [ ]:

# Usage in your notebook:
# After building the timeline:
# timeline = builder.build_from_xdf_files(xdf_file_paths=demo_xdf_paths)

# Export to JSON:
from pathlib import Path
output_path = Path("timeline_export.json")
export_timeline_to_json(timeline, output_path, include_raw_data=True, max_samples_per_stream=10000)

### to AirTable

In [ ]:
# ===================================================================================
# Complete Airtable Export Demonstration
# Export the newest EEG dataset from loaded XDF files to Airtable
# ===================================================================================

from phopymnehelper.exporters.AiirTable_Exporter import export_eeg_dataset_to_airtable
from pathlib import Path
import pandas as pd

# ===================================================================================
# Step 1: Ensure data is loaded (assuming this was already done in previous cells)
# ===================================================================================
# The following variables should already exist from earlier cells:
# - _out_eeg_raw: List of MNE Raw objects (sorted by time, newest last)
# - _out_xdf_stream_infos_df: DataFrame with stream information
# - lab_recorder_xdf_files: List of XDF file paths

# Verify data is loaded
assert '_out_eeg_raw' in locals() or '_out_eeg_raw' in globals(), "Please load XDF data first using LabRecorderXDF.load_and_process_all()"
assert len(_out_eeg_raw) > 0, "No EEG datasets loaded"

# ===================================================================================
# Step 2: Get the newest EEG dataset
# ===================================================================================
# The _out_eeg_raw list is sorted by time (most recent last), so the newest is the last item
newest_eeg_raw = _out_eeg_raw[-1]

# Get the corresponding stream info for the newest dataset
# Find the dataset index (should be the highest xdf_dataset_idx)
newest_dataset_idx = _out_xdf_stream_infos_df['xdf_dataset_idx'].max()
newest_stream_info = _out_xdf_stream_infos_df[_out_xdf_stream_infos_df['xdf_dataset_idx'] == newest_dataset_idx].iloc[0]

# Get the source XDF file path
newest_xdf_filename = newest_stream_info.get('xdf_filename', None)
newest_xdf_file_path = None
if newest_xdf_filename:
    # Find the matching file in lab_recorder_xdf_files
    for xdf_file in lab_recorder_xdf_files:
        if xdf_file.name == newest_xdf_filename:
            newest_xdf_file_path = xdf_file
            break

# If not found, try to get from raw.info.description
if newest_xdf_file_path is None:
    desc = newest_eeg_raw.info.get('description', None)
    if desc:
        newest_xdf_file_path = Path(desc)

print(f"Newest EEG Dataset Info:")
print(f"  Dataset Index: {newest_dataset_idx}")
print(f"  Recording Date: {newest_eeg_raw.info.get('meas_date', 'N/A')}")
print(f"  Duration: {newest_eeg_raw.times[-1]:.2f} seconds" if len(newest_eeg_raw.times) > 0 else "  Duration: N/A")
print(f"  Channels: {len(newest_eeg_raw.info['ch_names'])}")
print(f"  Source File: {newest_xdf_file_path}")

# ===================================================================================
# Step 3: Prepare Airtable credentials
# ===================================================================================
# Set your Airtable credentials (you may want to use environment variables or a config file)
AIRTABLE_API_KEY = "your_airtable_api_key_here"  # Get from https://airtable.com/api
AIRTABLE_BASE_ID = "appXXXXXXXXXXXXXX"  # Get from your Airtable base URL
AIRTABLE_TABLE_NAME = "EEG Recordings"  # Name of your table in Airtable

# ===================================================================================
# Step 4: Prepare additional fields from stream info
# ===================================================================================
additional_fields = {
    'XDF Filename': newest_stream_info.get('xdf_filename', None),
    'Stream Name': newest_stream_info.get('name', None),
    'Stream Type': newest_stream_info.get('type', None),
    'Sampling Rate (Hz)': newest_stream_info.get('fs', None),
    'Number of Samples': newest_stream_info.get('n_samples', None),
    'Recording Day Date': newest_stream_info.get('recording_day_date', None),
    'Duration (seconds)': newest_stream_info.get('duration_sec', None),
    'Source ID': newest_stream_info.get('source_id', None),
    'Hostname': newest_stream_info.get('hostname', None),
    'Device Key': newest_stream_info.get('eeg_device_key', None),
    'Number of Segments': newest_stream_info.get('n_eeg_segments_in_group', None),
}

# Remove None values
additional_fields = {k: v for k, v in additional_fields.items() if v is not None}

# Convert datetime/timedelta objects to strings if needed
for key, value in additional_fields.items():
    if hasattr(value, 'isoformat'):
        additional_fields[key] = value.isoformat()
    elif hasattr(value, 'total_seconds'):
        additional_fields[key] = value.total_seconds()

# ===================================================================================
# Step 5: Export to Airtable
# ===================================================================================
print(f"\nExporting to Airtable...")
print(f"  Base ID: {AIRTABLE_BASE_ID}")
print(f"  Table: {AIRTABLE_TABLE_NAME}")

result = export_eeg_dataset_to_airtable(
    raw=newest_eeg_raw,
    airtable_base_id=AIRTABLE_BASE_ID,
    airtable_table_name=AIRTABLE_TABLE_NAME,
    airtable_api_key=AIRTABLE_API_KEY,
    xdf_file_path=newest_xdf_file_path,
    additional_fields=additional_fields
)

# ===================================================================================
# Step 6: Display results
# ===================================================================================
if result['success']:
    print(f"\n✅ Successfully exported to Airtable!")
    print(f"  Record ID: {result['record_id']}")
    print(f"  Fields created: {list(result['fields'].keys())}")
    
    # Display some key fields
    print(f"\nKey exported fields:")
    key_fields = ['Recording Date', 'File Name', 'Number of Channels', 
                  'Sampling Rate (Hz)', 'Duration (seconds)', 'Stream Name']
    for field_name in key_fields:
        if field_name in result['fields']:
            print(f"  {field_name}: {result['fields'][field_name]}")
else:
    print(f"\n❌ Export failed!")
    print(f"  Error: {result.get('error', 'Unknown error')}")

# ===================================================================================
# Optional: Export multiple newest datasets (e.g., last 5)
# ===================================================================================
# If you want to export the last N newest datasets:
"""
from phopymnehelper.exporters.AiirTable_Exporter import export_multiple_eeg_datasets_to_airtable

# Get last 5 newest datasets
n_newest = 5
newest_raws = _out_eeg_raw[-n_newest:]
newest_indices = _out_xdf_stream_infos_df['xdf_dataset_idx'].nlargest(n_newest).tolist()
newest_xdf_paths = []

for idx in newest_indices:
    stream_info = _out_xdf_stream_infos_df[_out_xdf_stream_infos_df['xdf_dataset_idx'] == idx].iloc[0]
    xdf_filename = stream_info.get('xdf_filename', None)
    if xdf_filename:
        for xdf_file in lab_recorder_xdf_files:
            if xdf_file.name == xdf_filename:
                newest_xdf_paths.append(xdf_file)
                break

# Export all
results = export_multiple_eeg_datasets_to_airtable(
    raws=newest_raws,
    airtable_base_id=AIRTABLE_BASE_ID,
    airtable_table_name=AIRTABLE_TABLE_NAME,
    airtable_api_key=AIRTABLE_API_KEY,
    xdf_file_paths=newest_xdf_paths
)

# Check results
successful = sum(1 for r in results if r['success'])
print(f"Successfully exported {successful}/{len(results)} datasets")
"""

In [ ]:
enable_hide_extra_track_x_axes = False
if len(timeline.ui.matplotlib_view_widgets) > 1:
    # Get all plot items
    all_plot_items = []
    for widget_name, widget in timeline.ui.matplotlib_view_widgets.items():
        plot_item = widget.getRootPlotItem()
        if plot_item is not None:
            all_plot_items.append((widget_name, plot_item))
    
    # Hide x-axis for all except the last one (bottom-most)
    if len(all_plot_items) > 1:
        # Hide x-axis for all tracks except the last one
        if enable_hide_extra_track_x_axes:
            for widget_name, plot_item in all_plot_items[:-3]:
                plot_item.hideAxis('bottom')
            # Ensure the last track shows its x-axis
            all_plot_items[-1][1].showAxis('bottom')
        else:
            ## show all
            for widget_name, plot_item in all_plot_items:
                plot_item.showAxis('bottom')



In [ ]:
timeline.track_datasources['EEG_Epoc X'].zo #['Epoc X']


In [ ]:
timeline.track_renderers['EEG_Epoc X']

In [ ]:
zoom_to_fit

In [ ]:
timeline.remove_track(video_track_name)

# SavedSessionProcessor

In [ ]:

sso: SavedSessionsProcessor = SavedSessionsProcessor(eeg_recordings_file_path=eeg_recordings_file_path,
                                                     headset_motion_recordings_file_path=headset_motion_recordings_file_path, WhisperVideoTranscripts_LSL_Converted_file_path=WhisperVideoTranscripts_LSL_Converted, pho_log_to_LSL_recordings_path=pho_log_to_LSL_recordings_path,
                                                    eeg_analyzed_parent_export_path=eeg_analyzed_parent_export_path, 
                                                     n_most_recent_sessions_to_preprocess=n_most_recent_sessions_to_preprocess, 
                                                    # should_load_data=False, should_load_preprocessed=False,
                                                    should_load_data=True, should_load_preprocessed=False,
                                                    #  should_load_data=True, should_load_preprocessed=True,
													)

In [ ]:
most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()
demo_xdf_paths: List[Path] = [Path(v) for v in most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()]

modern_found_EEG_recording_file_df = most_recent_modern_found_EEG_recording_file_df

In [ ]:
modern_found_EEG_recording_file_df.sort_values(by='meas_datetime', ascending=False, inplace=False, ignore_index=True, na_position='last')

In [ ]:
most_recent_xdfs = [Path(v).resolve() for v in deepcopy(modern_found_EEG_recording_file_df).head(n_most_recent_sessions_to_preprocess)['src_file'].tolist()]
most_recent_xdfs

In [ ]:
# modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=eeg_recordings_file_path)
# modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=modern_found_EEG_recording_files)

In [ ]:
# updated_file_paths, (pending_updated_recording_file_df, modern_found_EEG_recording_file_df, pre_processed_EEG_recording_file_df) = HistoricalData.discover_updated_recording_files(eeg_recordings_file_path=sso.eeg_recordings_file_path,
#                                                                                                                                                                                    eeg_analyzed_parent_export_path=sso.eeg_analyzed_parent_export_path)


In [ ]:
# included_xdf_file_names = [
# 	"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-21T051157.400Z_eeg.xdf", ## When it started to work
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-20T215045.162Z_eeg.xdf"
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-20T164055.381Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-18T092615.398Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-17T215112.606Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-17T124127.644Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-17T214946.083Z_eeg.xdf",
#     # # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-09-22T182649.051Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T220233.548Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T212744.771Z_eeg.xdf",
#     # # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T212721.939Z_eeg.xdf",
#     # # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T212528.076Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-09-23T141026.412Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-09-22T213547.659Z_eeg.xdf",
# ]

included_xdf_file_names = deepcopy(most_recent_xdfs)

included_xdf_file_names = [Path(v).resolve() for v in included_xdf_file_names]
included_xdf_file_names = [v.name for v in included_xdf_file_names]


# included_xdf_file_names = None ## include all 
included_xdf_file_names

# 2025-09-18 - LabRecorder XDF Imports

In [ ]:

from phoofflineeeganalysis.analysis.xdf_files import LabRecorderXDF, XDFDataStreamAccessor


labRecorder_PostProcessed_path: Path = sso.eeg_analyzed_parent_export_path.joinpath(f'LabRecorder_PostProcessed')
labRecorder_PostProcessed_path.mkdir(exist_ok=True)

should_load_full_file_data: bool = True
# should_load_full_file_data: bool = False
# should_write_final_merged_eeg_fif: bool = True
should_write_final_merged_eeg_fif: bool = False

fail_on_exception = False
# fail_on_exception = True

_out_eeg_raw, _out_xdf_stream_infos_df, lab_recorder_xdf_files = LabRecorderXDF.load_and_process_all(lab_recorder_output_path=lab_recorder_output_path, 
                                                                                                     labRecorder_PostProcessed_path=labRecorder_PostProcessed_path, 
                                                                                                     should_load_full_file_data=should_load_full_file_data, should_write_final_merged_eeg_fif=should_write_final_merged_eeg_fif,
                                                                                                     included_xdf_file_names=included_xdf_file_names, fail_on_exception=fail_on_exception)
xdf_dataset_indicies = np.unique(deepcopy(_out_xdf_stream_infos_df).reset_index(drop=False, inplace=False)['xdf_dataset_idx'].to_numpy())
n_unique_xdf_datasets: int = len(xdf_dataset_indicies)
print(f'n_unique_xdf_datasets: {n_unique_xdf_datasets}')
_out_xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df, active_only_out_eeg_raws=_out_eeg_raw) # [_out_xdf_stream_infos_df['name'] == 'Epoc X']
_out_xdf_stream_infos_df


## 2m 30s

## Timeline

In [ ]:
included_dataset_idxs = np.arange(3)

_active_out_xdf_stream_infos_df = _out_xdf_stream_infos_df[np.isin(_out_xdf_stream_infos_df['xdf_dataset_idx'], included_dataset_idxs)]
_active_out_eeg_raw = [_out_eeg_raw[i] for i in included_dataset_idxs]

_active_out_xdf_stream_infos_df
len(_active_out_eeg_raw)

In [ ]:
import importlib
import pypho_timeline.rendering.datasources.specific.eeg as eeg
import pypho_timeline.rendering.datasources.track_datasource as track_datasource
import pypho_timeline.rendering.graphics.track_renderer as track_renderer
import pypho_timeline.rendering.async_detail_fetcher as async_detail_fetcher

importlib.reload(track_datasource)
importlib.reload(eeg)
importlib.reload(track_renderer)
importlib.reload(async_detail_fetcher)


In [ ]:

import pyphoplacecellanalysis.External.pyqtgraph as pg
from pypho_timeline.timeline_builder import TimelineBuilder

## INPUTS: _out_eeg_raw, _out_xdf_stream_infos_df -- builds and displays timeline with proper datetime support

# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

builder: TimelineBuilder = TimelineBuilder()
# timeline = builder.build_from_eeg_raw_and_stream_info(eeg_raws=_out_eeg_raw, stream_infos_df=_out_xdf_stream_infos_df) 
timeline = builder.build_from_eeg_raw_and_stream_info(eeg_raws=_active_out_eeg_raw, stream_infos_df=_active_out_xdf_stream_infos_df) 

In [ ]:
# timeline.interval_datasource_names
timeline.track_datasources

In [ ]:
from pypho_timeline.utils.datetime_helpers import datetime_to_unix_timestamp

x0 = datetime_to_unix_timestamp(timeline.total_data_start_time)
x1 = datetime_to_unix_timestamp(timeline.total_data_end_time)

for plot in timeline.interval_rendering_plots:
    plot.setXRange(x0, x1, padding=0)

In [ ]:
timeline.
# timeline.active_window_visible_intervals_dict

In [ ]:
## I want to pass _out_eeg_raw and _out_xdf_stream_infos_df to the timeline


In [ ]:
_out_eeg_raw[-1].annotations.to_data_frame('datetime')

In [ ]:
(1435.384078 / 60.0) # ~24 mins

(3884.001499 / 60.0) # ~65 mins


In [ ]:
# ## INPUTS to timeline: _out_xdf_stream_infos_df: pd.DataFrame, _out_eeg_raw, lab_recorder_xdf_files, xdf_dataset_indicies

# from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
#     TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
#     MotionRecordingTrack, PhoLogTrack, WhisperTrack
# )

# timeline = TimelineWidget()
# timeline.add_tracks_from_xdf_streams(_out_xdf_stream_infos_df)
# timeline.show()

In [ ]:
# from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
#     TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
#     MotionRecordingTrack, PhoLogTrack, WhisperTrack
# )

# timeline = TimelineWidget()
# timeline.add_tracks_from_xdf_streams(_out_xdf_stream_infos_df)
# timeline.show()

In [ ]:
## sort by 'recording_day_date'
from datetime import timedelta


_filtering_xdf_stream_infos_df = deepcopy(_out_xdf_stream_infos_df).reset_index(drop=True)
_all_xdf_filenames = list(set(_filtering_xdf_stream_infos_df['xdf_filename'].to_list())) ## set(...) to de-duplicate the list

# _out_xdf_stream_infos_df['recording_day_date']
is_eeg_stream = (_filtering_xdf_stream_infos_df['type'] == 'EEG') ## only EEG files
_filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df[is_eeg_stream].reset_index(drop=True)
# _filtering_xdf_stream_infos_df
# is_sufficiently_long = [(_filtering_xdf_stream_infos_df['duration_sec'] > timedelta(seconds=30.0))]
# _filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df[is_sufficiently_long].reset_index(drop=True)
_filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df[(_filtering_xdf_stream_infos_df['duration_sec'] > timedelta(seconds=90.0))].reset_index(drop=True)

_filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df.sort_values(by=['recording_datetime', 'created_at_dt', 'first_timestamp_dt', 'last_timestamp_dt'], ascending=True, inplace=False)
_filtering_xdf_stream_infos_df
# _out_xdf_stream_infos_df[is_sufficiently_long]


In [ ]:
_all_xdf_filenames

In [ ]:
included_xdf_filenames: List[str] = _filtering_xdf_stream_infos_df['xdf_filename'].to_list()
excluded_xdf_filenames: List[str] = list(set(_all_xdf_filenames) - set(included_xdf_filenames))
excluded_xdf_filenames

# included_xdf_filenames = ['LabRecorder_2025-09-10T153731.079Z_eeg.xdf',
#  'LabRecorder_2025-09-11T014154.084Z_eeg.xdf',
#  'LabRecorder_2025-09-11T101328.256Z_eeg.xdf',
#  'LabRecorder_2025-09-11T154549.460Z_eeg.xdf',
#  'LabRecorder_2025-09-12T220903.464Z_eeg.xdf',
#  'LabRecorder_2025-09-18T031842.989Z_eeg.xdf',
#  'LabRecorder_2025-09-18T121337.267Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-12T215537.892Z.xdf',
#  'LabRecorder_Apogee_2025-09-18T151839.043Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-18T152308.395Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-19T051346.012Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-19T205118.364Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-20T214749.964Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-21T003051.428Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-21T085541.696Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-22T213547.659Z_eeg.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-12T014018.162Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-13T021042.451Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-13T031209.598Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T001746.449Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T012938.518Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T015439.979Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T022210.910Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-20T023803.932Z.xdf']




# ['xdf_filename', 'first_timestamp', 'last_timestamp', 'sample_count',
    #    'lab_recorder_xdf_file_idx', 'xdf_filename', 'proccessed_fif_filename',
    #    'proccessed_mat_filename', 'xdf_dataset_idx', 'recording_datetime',
    #    'recording_day_date', 'duration_sec']

In [ ]:
_out_xdf_stream_infos_df.reset_index(drop=True).groupby('xdf_dataset_idx').first()

In [ ]:
lab_recorder_xdf_files

In [ ]:


def extract_annotations_df(active_only_out_eeg_raws) -> pd.DataFrame:
    ## Extract comments/notes/annotations/etc from the outputs

    _extracted_comments = []
    ignored_comment_descriptions = ['BAD_motion', '']
    for a_raw in active_only_out_eeg_raws:
        an_annotations = a_raw.annotations
        if (an_annotations is not None) and (len(an_annotations) > 0):
            an_annotation_df = an_annotations.to_data_frame(time_format='datetime')
            an_annotation_df = an_annotation_df[np.logical_not(np.isin(an_annotation_df['description'], ignored_comment_descriptions))]
            _extracted_comments.append(an_annotation_df)
            # an_annotation_df


    extracted_comments_df: pd.DataFrame = pd.concat(_extracted_comments)
    extracted_comments_df = extracted_comments_df.rename(columns={'onset':'time', 'description':'text'}, inplace=False)
    return extracted_comments_df

annotations_df = extract_annotations_df(_out_eeg_raw)
annotations_df

In [ ]:
# from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.SpikeRasterWidgets.Spike2DRaster import Spike2DRaster, SynchronizedPlotMode

In [ ]:
_out_xdf_stream_infos_df

In [ ]:
len(_out_xdf_stream_infos_df)
_out_xdf_stream_infos_df.loc[0]

In [ ]:
_out_eeg_raw

# 2025-12-10 - Build Timeline Browser from parsed streams

_out_xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df, active_only_out_eeg_raws=_out_eeg_raw) # [_out_xdf_stream_infos_df['name'] == 'Epoc X']
_out_xdf_stream_infos_df

In [ ]:
## INPUTS: _out_xdf_stream_infos_df: pd.DataFrame

In [ ]:
_out_xdf_stream_infos_df

# csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
csv_save_path = Path('../output').joinpath('2025-12-17_all_xdf_stream_infos.csv').resolve()
print(f'csv_save_path: "{csv_save_path.as_posix()}"')
_out_xdf_stream_infos_df.to_csv(csv_save_path)

## load with: 
# csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
# assert csv_save_path.exists()
# all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


In [ ]:
# from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
#     TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
#     MotionRecordingTrack, PhoLogTrack, WhisperTrack
# )

timeline = TimelineWidget()


csv_save_path = Path('../output').joinpath('2025-12-09_parsed_videos.csv').resolve()
assert csv_save_path.exists()
video_df: pd.DataFrame = pd.read_csv(csv_save_path)

csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
assert csv_save_path.exists()
all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


# Add tracks from different modalities
timeline.add_track(VideoMetadataTrack(video_df))
timeline.add_tracks_from_xdf_streams(all_xdf_stream_infos_df)
timeline.show()

In [ ]:
timeline.add_tracks_from_xdf_streams(all_xdf_stream_infos_df)

In [ ]:

timeline.add_track(EEGRecordingTrack(eeg_df))
timeline.add_track(MotionRecordingTrack(motion_df))
timeline.add_track(PhoLogTrack(pho_log_df))
timeline.add_track(WhisperTrack(whisper_df))

# 2025-12-11 - Build Detailed Timeline Browser from actual recent data

In [ ]:
## INPUTS: _out_xdf_stream_infos_df: pd.DataFrame
from phopylslhelper.file_metadata_caching.video_metadata import VideoMetadataParser

output_folder = Path('../output').resolve()
assert output_folder.exists()

# csv_save_path = output_folder.joinpath('2025-12-17_parsed_videos.csv').resolve()
# assert csv_save_path.exists()
# video_df: pd.DataFrame = pd.read_csv(csv_save_path)
# video_df


## parse videos from scratch because it's pretty fast:
video_recordings_folder_path = Path(r"M:\ScreenRecordings\EyeTrackerVR_Recordings")

print(f"Parsing videos in: {video_recordings_folder_path}")
video_df = VideoMetadataParser.parse_video_folder(video_recordings_folder_path)
video_df

In [3]:
from phopylslhelper.file_metadata_caching.manager import BaseFileMetadataManager
from phopylslhelper.file_metadata_caching.video_metadata import VideoMetadataParser
from phopylslhelper.file_metadata_caching.file_metadata import BaseFileMetadataParser

video_manager: BaseFileMetadataManager = BaseFileMetadataManager(parse_folders=[Path("M:/ScreenRecordings/EyeTrackerVR_Recordings"), Path("M:/ScreenRecordings/REC_continuous_video_recorder")],
                                                                parsers={'video': VideoMetadataParser},)
video_manager.metadata_df

,video_start_datetime,video_duration,video_end_datetime,video_num_frames,video_fps,video_width,video_height,video_file_path,video_file_size,cache_file_size,cache_file_mtime
3336,2026-04-01 06:13:23,3599.900000,2026-04-01 07:13:22.900000,107997,30.0,640,480,M:\ScreenRecordings\REC_continuous_video_recor...,113802519,113802519,1.775042e+09
3337,2026-04-01 06:13:23,3599.900000,2026-04-01 07:13:22.900000,107997,30.0,640,480,M:\ScreenRecordings\REC_continuous_video_recor...,113802519,113802519,1.775042e+09
3334,2026-04-01 04:59:24,2698.000000,2026-04-01 05:44:22.000000,80940,30.0,640,480,M:\ScreenRecordings\REC_continuous_video_recor...,91708331,91708331,1.775037e+09
3335,2026-04-01 04:59:24,2698.000000,2026-04-01 05:44:22.000000,80940,30.0,640,480,M:\ScreenRecordings\REC_continuous_video_recor...,91708331,91708331,1.775037e+09
3332,2026-04-01 03:59:24,3599.966667,2026-04-01 04:59:23.966667,107999,30.0,640,480,M:\ScreenRecordings\REC_continuous_video_recor...,129642652,129642652,1.775034e+09
...,...,...,...,...,...,...,...,...,...,...,...
5,2024-09-04 20:05:10,377.133333,2024-09-04 20:11:27.133333,11314,30.0,176,176,M:\ScreenRecordings\EyeTrackerVR_Recordings\20...,15233317,15233317,1.725495e+09
2,2024-09-04 11:35:54,2656.333333,2024-09-04 12:20:10.333333,79690,30.0,176,176,M:\ScreenRecordings\EyeTrackerVR_Recordings\20...,64725221,64725221,1.725467e+09
3,2024-09-04 11:35:54,2656.333333,2024-09-04 12:20:10.333333,79690,30.0,176,176,M:\ScreenRecordings\EyeTrackerVR_Recordings\20...,64725221,64725221,1.725467e+09
0,2023-02-09 15:53:04,17.733333,2023-02-09 15:53:21.733333,1064,60.0,1920,1080,M:\ScreenRecordings\EyeTrackerVR_Recordings\20...,923621,923621,1.675976e+09


In [ ]:

from phoofflineeeganalysis.analysis.UI.timeline import (
    TimelineWidget,
    VideoMetadataTrack,
)


timeline = TimelineWidget()

timeline.add_track(VideoMetadataTrack(video_df))
timeline.show()

In [ ]:
_out_xdf_stream_infos_df

csv_save_path = output_folder.joinpath('2025-12-15_all_xdf_stream_infos.csv').resolve()
print(f'csv_save_path: "{csv_save_path.as_posix()}"')
_out_xdf_stream_infos_df.to_csv(csv_save_path)

## load with: 
# csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
# assert csv_save_path.exists()
# all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


In [ ]:
csv_save_path = Path('../output').joinpath('2025-12-15_all_xdf_stream_infos.csv').resolve()
print(f'csv_save_path: "{csv_save_path.as_posix()}"')
_out_xdf_stream_infos_df.to_csv(csv_save_path)

In [ ]:
from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
    TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
    MotionRecordingTrack, PhoLogTrack, WhisperTrack
)

timeline = TimelineWidget()


csv_save_path = Path('../output').joinpath('2025-12-09_parsed_videos.csv').resolve()
assert csv_save_path.exists()
video_df: pd.DataFrame = pd.read_csv(csv_save_path)

csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
assert csv_save_path.exists()
all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


# Add tracks from different modalities
timeline.add_track(VideoMetadataTrack(video_df))
timeline.add_tracks_from_xdf_streams(all_xdf_stream_infos_df)
timeline.show()

# From detailed data: 
INPUTS to timeline: _out_xdf_stream_infos_df: pd.DataFrame, _out_eeg_raw, lab_recorder_xdf_files, xdf_dataset_indicies


In [ ]:
## INPUTS: active_only_out_eeg_raws, results
## INPUTS: extracted_comments_df: pd.DataFrame ## comments track

In [ ]:
_out_xdf_stream_infos_df

In [ ]:
# lab_recorder_output_path

# print(list(_out_xdf_stream_infos_df.columns))

if 'xdf_file_path' not in _out_xdf_stream_infos_df.columns:
    _out_xdf_stream_infos_df['xdf_file_path'] = _out_xdf_stream_infos_df['xdf_filename'].map(lambda x: Path(lab_recorder_output_path).joinpath(x).resolve())


xdf_file_paths: List[Path] = _out_xdf_stream_infos_df['xdf_file_path'].to_list()
xdf_file_paths



In [ ]:
## INPUTS to timeline: _out_xdf_stream_infos_df: pd.DataFrame, _out_eeg_raw, lab_recorder_xdf_files, xdf_dataset_indicies
from phoofflineeeganalysis.analysis.UI.timeline.datasource.datasources import XDFDatasource

a_ds = XDFDatasource(a_xdf_file=xdf_file_paths[0], datasource_name='test_xdf')
a_ds

In [ ]:
# a_ds.df
# list(a_ds.lab_recorder_xdf.stream_infos.columns)

a_ds.lab_recorder_xdf.stream_infos
# 'last_timestamp_dt'

In [ ]:
a_ds.total_datasource_start_end_times

In [ ]:

from phoofflineeeganalysis.analysis.UI.timeline import (
    TimelineWidget, EEGRecordingTrack,
    XDFStreamTrack
)

## INPUTS: xdf_file_paths
timeline = TimelineWidget()

# Get stream information DataFrame from the datasource
stream_infos_df = a_ds.lab_recorder_xdf.stream_infos

# Add tracks for all available stream modalities
if stream_infos_df is not None and not stream_infos_df.empty:
    timeline.add_tracks_from_xdf_streams(stream_infos_df, fail_on_exception=False)

# Always add XDFStreamTrack in addition to individual modality tracks
timeline.add_track(track=XDFStreamTrack(a_ds))

timeline.show()

In [ ]:
a_ds.get_detailed_data()

In [ ]:
# a_ds.lab_recorder_xdf
a_ds.lab_recorder_xdf.datasets

In [ ]:
a_ds.lab_recorder_xdf.stream_infos

In [ ]:
from phoofflineeeganalysis.analysis.UI.timeline.tracks.MotionRecordingTrack import MotionRecordingTrack
from phoofflineeeganalysis.analysis.MNE_helpers import up_convert_raw_objects
from phoofflineeeganalysis.analysis.MNE_helpers import MNEHelpers

## INPUTS: _out_xdf_stream_infos_df
a_motion_raw = up_convert_raw_objects(a_ds.lab_recorder_xdf.datasets_dict[DataModalityType.MOTION.value])[0]
a_motion_df = a_ds.lab_recorder_xdf.streams_timestamp_dfs['Epoc X Motion'] ## not right
a_motion_overview_df = a_ds.lab_recorder_xdf.stream_infos ## overview
dataset_MOTION_df = a_motion_raw.to_data_frame(time_format='datetime')
dataset_MOTION_df = MNEHelpers.convert_df_columns_to_datetime(dataset_MOTION_df, dt_col_names=["start_time", "end_time"])
dataset_MOTION_df
from phoofflineeeganalysis.analysis.UI.timeline.datasource.datasources import IntervalDataframeDatasource

motion_ds = IntervalDataframeDatasource(df=dataset_MOTION_df, time_column_name='time') # , time_col_name='time'
motion_ds
# def load_motion_series(metadata: Dict[str, Any], window_ts: Tuple[float, float]) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
#     # Use metadata['xdf_filename'], metadata['sampling_rate'], etc.
#     # Return: {"AccX": (t_accx, v_accx), "AccY": (...), ..., "GyroZ": (...)}
#     xdf_filepath = Path(metadata['xdf_filename']).resolve() ## load the detailed data from the XDF file
    
motion_track = MotionRecordingTrack(motion_source=motion_ds, height=80)
timeline.add_track(motion_track)

In [ ]:
motion_track._is_detailed_mode
# motion_track._ensure_detailed_items()
motion_track._render_detailed(motion_ds.total_datasource_start_end_times)
motion_track._is_detailed_mode

In [ ]:
motion_track._is_detailed_mode
motion_track.set_detailed_threshold(seconds=1000000.0)  # Adjust this value as needed
motion_track.update_display()
motion_track._is_detailed_mode

In [ ]:
motion_track._ensure_detailed_items()

In [ ]:
motion_ds.total_datasource_start_end_times
motion_ds.total_df_start_end_times


In [ ]:
timeline.set_time_range(start_dt=motion_ds.total_datasource_start_end_times[0], end_dt=motion_ds.total_datasource_start_end_times[1])

In [ ]:
# dataset_MOTION_df ## overview

# a_ds.lab_recorder_xdf.stream_infos ## overview
a_motion_raw[0].to_df()

## Other Tracks

In [ ]:
# Build the detailed PhoLogger track from `extracted_comments_df: pd.DataFrame` with columns: ['time', 'text']. It should componently layout the strings so they don't excessively overlap, elliding or wrapping when needed.
## It can use the full height to stagger strings that would otherwise overlap horizontally.
from phoofflineeeganalysis.analysis.UI.timeline import PhoLogTrack, WhisperTrack
## Extract comments/notes/annotations/etc from the outputs (`active_only_out_eeg_raws`)

_extracted_comments = []
ignored_comment_descriptions = ['BAD_motion', '']
for a_raw in active_only_out_eeg_raws:
    an_annotations = a_raw.annotations
    if (an_annotations is not None) and (len(an_annotations) > 0):
        an_annotation_df = an_annotations.to_data_frame(time_format='datetime')
        an_annotation_df = an_annotation_df[np.logical_not(np.isin(an_annotation_df['description'], ignored_comment_descriptions))]
        _extracted_comments.append(an_annotation_df)
        # an_annotation_df


extracted_comments_df: pd.DataFrame = pd.concat(_extracted_comments)
extracted_comments_df = extracted_comments_df.rename(columns={'onset':'time', 'description':'text'}, inplace=False)
extracted_comments_df

timeline.add_track(PhoLogTrack(extracted_comments_df))

In [ ]:
from phoofflineeeganalysis.analysis.MNE_helpers import up_convert_raw_obj
from phopymnehelper.analysis.computations.EEG_data import EEGData
from PhoOfflineEEGAnalysis.src.phoofflineeeganalysis.analysis.SavedSessionsProcessor import LabRecorderXDF


assert lab_recorder_output_path.exists()

lab_recorder_xdf_files: List[Path] = list(lab_recorder_output_path.glob('*.xdf'))
n_total_found_files: int = len(lab_recorder_xdf_files)
if included_xdf_file_names is not None:
    print(f'limiting to included_xdf_file_names: {included_xdf_file_names}...')
    lab_recorder_xdf_files = [v for v in lab_recorder_xdf_files if v.name in included_xdf_file_names]
    n_filtered_found_files: int = len(lab_recorder_xdf_files)
    print(f'\tlimited to {n_filtered_found_files}/{n_total_found_files} files')

if not should_load_full_file_data:
    assert (not should_write_final_merged_eeg_fif)

if (labRecorder_PostProcessed_path is not None) and should_write_final_merged_eeg_fif:
    labRecorder_PostProcessed_path.mkdir(exist_ok=True)

# a_xdf_file = lab_recorder_xdf_files[-3]
# a_xdf_file = lab_recorder_xdf_files[-1]
# a_xdf_file = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001\LabRecorder_2025-09-18T031842.989Z_eeg.xdf").resolve()
# a_xdf_file = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001\LabRecorder_2025-09-18T121337.267Z_eeg.xdf").resolve()

_out_eeg_raw = []
_out_xdf_stream_infos_df = []

for an_xdf_file_idx, a_xdf_file in enumerate(lab_recorder_xdf_files):
    print(f'trying to process XDF file {an_xdf_file_idx}/{len(lab_recorder_xdf_files)}: "{a_xdf_file.as_posix()}"...')
    try:
        _obj = LabRecorderXDF.init_from_lab_recorder_xdf_file(a_xdf_file=a_xdf_file, should_load_full_file_data=should_load_full_file_data, debug_print=True)
        stream_infos = _obj.stream_infos
        raws = _obj.datasets
        raws_dict = _obj.datasets_dict
        eeg_raws = raws_dict.get(DataModalityType.EEG.value, [])
        if len(eeg_raws) > 0:
            print(f'\tWARN: no EEG streams found in "{a_xdf_file.as_posix()}". Skipping file.')
            # Merge by device so we can handle multiple EEG streams per XDF
            merged_eeg_raws, merge_meta = LabRecorderXDF.merge_eeg_streams_by_device(
                eeg_raws=eeg_raws, strict_merge=False, debug_print=False
            )
            eeg_raw = up_convert_raw_obj(eeg_raw)
            EEGData.set_montage(datasets_EEG=[eeg_raw])
            eeg_raw.debug_test_annotations_timestamps()
            _out_eeg_raw.append(eeg_raw)
        
    except (ValueError, KeyError, AssertionError, TypeError) as e:
        print(f'\t failed with error: {e}\n\tskipping file.')
        if fail_on_exception:
            raise
        else:
            continue
        
    except Exception as e:
        print(f'\t failed with error: {e}\n\tskipping file.')
        raise



In [ ]:
motion_df = _out_xdf_stream_infos_df[_out_xdf_stream_infos_df['name'] == 'Epoc X Motion']
list(motion_df.columns) # 'xdf_filename', 'xdf_dataset_idx'
motion_df['xdf_filename']

In [ ]:
def load_motion_series(metadata: Dict[str, Any], window_ts: Tuple[float, float]) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    # Use metadata['xdf_filename'], metadata['sampling_rate'], etc.
    # Return: {"AccX": (t_accx, v_accx), "AccY": (...), ..., "GyroZ": (...)}
    xdf_filepath = Path(metadata['xdf_filename']).resolve() ## load the detailed data from the XDF file
    

DataModalityType.MOTION.value
## INPUTS: _out_xdf_stream_infos_df
motion_df = _out_xdf_stream_infos_df[_out_xdf_stream_infos_df['name'] == 'Epoc X Motion']
motion_track = MotionRecordingTrack(motion_df, detailed_data_provider=load_motion_series)
timeline.add_track(motion_track)

In [ ]:

timeline.add_track(EEGRecordingTrack(eeg_df))
timeline.add_track(MotionRecordingTrack(motion_df))
timeline.add_track(PhoLogTrack(pho_log_df))
timeline.add_track(WhisperTrack(whisper_df))

# 2025-01-05 - `pyPhoTimeline`

In [ ]:
from pypho_timeline.rendering.datasources.track_datasource import BaseTrackDatasource
from pypho_timeline.rendering.detail_renderers import PositionPlotDetailRenderer, VideoThumbnailDetailRenderer


class PositionTrackDatasource(BaseTrackDatasource):
    """Example TrackDatasource for position data.
    
    Inherits from BaseTrackDatasource and implements all required methods for
    displaying position data with async detail loading.
    """
    
    def __init__(self, position_df: pd.DataFrame, intervals_df: pd.DataFrame):
        """Initialize with position data and intervals.
        
        Args:
            position_df: DataFrame with columns ['t', 'x', 'y'] (or ['t', 'x'] for 1D)
            intervals_df: DataFrame with columns ['t_start', 't_duration'] for intervals
        """
        super().__init__()
        self.position_df = position_df
        self.intervals_df = intervals_df.copy()
        self.custom_datasource_name = "PositionTrack"
        
        # Add visualization columns to intervals
        self.intervals_df['series_vertical_offset'] = 0.0
        self.intervals_df['series_height'] = 1.0
        
        # Create pens and brushes
        color = pg.mkColor('blue')
        color.setAlphaF(0.3)
        pen = pg.mkPen(color, width=1)
        brush = pg.mkBrush(color)
        self.intervals_df['pen'] = [pen] * len(self.intervals_df)
        self.intervals_df['brush'] = [brush] * len(self.intervals_df)
    
    @property
    def df(self) -> pd.DataFrame:
        return self.intervals_df
    
    @property
    def time_column_names(self) -> list:
        return ['t_start', 't_duration', 't_end']
    
    @property
    def total_df_start_end_times(self) -> tuple:
        if len(self.intervals_df) == 0:
            return (0.0, 1.0)
        t_start = self.intervals_df['t_start'].min()
        t_end = (self.intervals_df['t_start'] + self.intervals_df['t_duration']).max()
        return (t_start, t_end)
    
    def get_updated_data_window(self, new_start: float, new_end: float) -> pd.DataFrame:
        """Get intervals overlapping with time window."""
        mask = (self.intervals_df['t_start'] + self.intervals_df['t_duration'] >= new_start) & \
               (self.intervals_df['t_start'] <= new_end)
        return self.intervals_df[mask].copy()
    
    def update_visualization_properties(self, dataframe_vis_columns_function):
        """Update visualization properties."""
        self.intervals_df = dataframe_vis_columns_function(self.intervals_df)
    
    def get_overview_intervals(self) -> pd.DataFrame:
        """Get overview intervals."""
        return self.intervals_df
    
    def fetch_detailed_data(self, interval: pd.Series) -> pd.DataFrame:
        """Fetch position data for an interval."""
        if self.position_df is None:
            return pd.DataFrame()  # Return empty DataFrame if no position data available
        t_start = interval['t_start']
        t_end = t_start + interval['t_duration']
        mask = (self.position_df['t'] >= t_start) & (self.position_df['t'] < t_end)
        return self.position_df[mask].copy()
    
    def get_detail_renderer(self):
        """Get detail renderer for position data."""
        if self.position_df is None:
            return PositionPlotDetailRenderer(pen_color='cyan', pen_width=2, y_column=None)
        return PositionPlotDetailRenderer(pen_color='cyan', pen_width=2, y_column='y' if 'y' in self.position_df.columns else None)
    
    def get_detail_cache_key(self, interval: pd.Series) -> str:
        """Get cache key for interval."""
        return f"position_{interval['t_start']:.3f}_{interval['t_duration']:.3f}"


class VideoTrackDatasource(BaseTrackDatasource):
    """Example TrackDatasource for video data.
    
    Inherits from BaseTrackDatasource and implements all required methods for
    displaying video intervals with async detail loading.
    """
    
    def __init__(self, video_intervals_df: pd.DataFrame):
        """Initialize with video intervals.
        
        Args:
            video_intervals_df: DataFrame with columns ['t_start', 't_duration', 'video_path']
        """
        super().__init__()
        self.video_intervals_df = video_intervals_df.copy()
        self.custom_datasource_name = "VideoTrack"
        
        # Add visualization columns
        self.video_intervals_df['series_vertical_offset'] = 0.0
        self.video_intervals_df['series_height'] = 50.0
        
        # Create pens and brushes
        color = pg.mkColor('green')
        color.setAlphaF(0.3)
        pen = pg.mkPen(color, width=1)
        brush = pg.mkBrush(color)
        self.video_intervals_df['pen'] = [pen] * len(self.video_intervals_df)
        self.video_intervals_df['brush'] = [brush] * len(self.video_intervals_df)
    
    @property
    def df(self) -> pd.DataFrame:
        return self.video_intervals_df
    
    @property
    def time_column_names(self) -> list:
        return ['t_start', 't_duration', 't_end']
    
    @property
    def total_df_start_end_times(self) -> tuple:
        if len(self.video_intervals_df) == 0:
            return (0.0, 1.0)
        t_start = self.video_intervals_df['t_start'].min()
        t_end = (self.video_intervals_df['t_start'] + self.video_intervals_df['t_duration']).max()
        return (t_start, t_end)
    
    def get_updated_data_window(self, new_start: float, new_end: float) -> pd.DataFrame:
        """Get intervals overlapping with time window."""
        mask = (self.video_intervals_df['t_start'] + self.video_intervals_df['t_duration'] >= new_start) & \
               (self.video_intervals_df['t_start'] <= new_end)
        return self.video_intervals_df[mask].copy()
    
    def update_visualization_properties(self, dataframe_vis_columns_function):
        """Update visualization properties."""
        self.video_intervals_df = dataframe_vis_columns_function(self.video_intervals_df)
    
    def get_overview_intervals(self) -> pd.DataFrame:
        """Get overview intervals."""
        return self.video_intervals_df
    
    def fetch_detailed_data(self, interval: pd.Series) -> dict:
        """Fetch video frames for an interval (simulated with random images)."""
        # In a real implementation, this would load video frames
        # For demo, generate synthetic frame data
        n_frames = max(1, int(interval['t_duration'] * 10))  # 10 fps
        frames = []
        for i in range(n_frames):
            # Generate a simple colored frame
            frame = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
            frames.append(frame)
        return {'frames': frames, 'timestamps': np.linspace(interval['t_start'], interval['t_start'] + interval['t_duration'], n_frames)}
    
    def get_detail_renderer(self):
        """Get detail renderer for video."""
        return VideoThumbnailDetailRenderer(thumbnail_height=50.0, spacing=0.1)
    
    def get_detail_cache_key(self, interval: pd.Series) -> str:
        """Get cache key for interval."""
        return f"video_{interval['t_start']:.3f}_{interval['t_duration']:.3f}"
